# **Notebook: 24H_CLEAN_PIPELINE.ipynb**

# Cell 1 — imports and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================================================
# 0. IMPORTS + PATHS
# =========================================================
import os
import re
import glob
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

AR_MAIN_BASE = "/content/drive/MyDrive/AR_Stratified"
HMI_BASE = f"{AR_MAIN_BASE}/HMI_SHARP"
GOES_BASE = f"{AR_MAIN_BASE}/GOES_XRAY_EVENTS"

# Final outputs
OUT_AR_CLEAN = f"{AR_MAIN_BASE}/HMI_AR_2010_2025_CLEAN_24H_BASE.csv"
OUT_AR_ML16  = f"{AR_MAIN_BASE}/HMI_AR_2010_2025_ML_READY_16_CLEAN_24H_BASE.csv"
OUT_24H_FINAL = f"{AR_MAIN_BASE}/HMI_AR_2010_2025_ML_READY_16_LABELED_MX1D_FLARE_FEATURES_24H_CLEAN.csv"

year_files = {
    2010: "hmi_2010_2010_fixed.csv",
    2011: "hmi_2011_2011_fixed.csv",
    2012: "hmi_2012_2012_fixed.csv",
    2013: "hmi_2013_2013_fixed.csv",
    2014: "hmi_2014_2014_fixed.csv",
    2015: "hmi_2015_2015_fixed.csv",
    2016: "hmi_2016_2016_fixed.csv",
    2017: "hmi_2017_2017_fixed.csv",
    2018: "hmi_2018_2018_fixed.csv",
    2019: "hmi_2019_2019_fixed.csv",
    2020: "hmi_2020_2020_fixed.csv",
    2021: "hmi_2021_2021_fixed.csv",
    2022: "hmi_2022_2022_fixed.csv",
    2023: "hmi_2023_2023_fixed.csv",
    2024: "hmi_2024_2024_fixed.csv",
    2025: "hmi_2025_2025_fixed.csv",
}

CORE16 = [
    "T_REC_dt", "year", "NOAA_AR",
    "MEANGBZ", "MEANGAM", "MEANGBT", "MEANGBH",
    "MEANJZD", "TOTUSJZ", "MEANALP", "MEANJZH",
    "ABSNJZH", "SAVNCPP", "MEANSHR", "SHRGT45",
    "R_VALUE", "USFLUX", "TOTPOT", "TOTUSJH"
]

print("Paths ready ✅")

Paths ready ✅


# Cell 2 — helpers for HMI cleaning

In [ ]:
# =========================================================
# 1. HMI HELPERS
# =========================================================
def parse_trec(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "T_REC" not in df.columns:
        raise ValueError("T_REC column not found in HMI file.")

    df["T_REC"] = df["T_REC"].astype(str).str.strip()

    try:
        df["T_REC_dt"] = pd.to_datetime(
            df["T_REC"],
            format="%Y.%m.%d_%H:%M:%S_TAI",
            errors="coerce"
        )
    except Exception:
        df["T_REC_dt"] = pd.to_datetime(
            df["T_REC"].str.replace("_TAI", "", regex=False),
            errors="coerce"
        )

    df["year"] = df["T_REC_dt"].dt.year
    return df


def extract_primary_ar(x):
    s = str(x).strip()
    if s.lower() in {"none", "nan", "missing", ""}:
        return np.nan
    m = re.search(r"\d+", s)
    return int(m.group()) if m else np.nan


def ensure_noaa(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "NOAA_AR" in df.columns:
        df["NOAA_AR"] = pd.to_numeric(df["NOAA_AR"], errors="coerce").astype("Int64")
    elif "NOAA_ARS" in df.columns:
        df["NOAA_AR"] = df["NOAA_ARS"].apply(extract_primary_ar).astype("Int64")
    else:
        df["NOAA_AR"] = pd.Series([pd.NA] * len(df), dtype="Int64")

    return df

# Cell 3 — rebuild one canonical HMI base from raw yearly files

In [ ]:
# =========================================================
# 2. REBUILD CANONICAL HMI BASE FROM RAW YEARLY FILES
# =========================================================
all_years = []

print("=== CLEAN 24H HMI BUILD STARTED ===")

for yr, fname in year_files.items():
    path = f"{HMI_BASE}/{fname}"
    print(f"\nLoading {yr}: {fname}")

    if not os.path.exists(path):
        print(f"⚠ File missing: {path}")
        continue

    df = pd.read_csv(path)
    df = parse_trec(df)
    df = ensure_noaa(df)

    # keep only intended year
    df = df[df["year"] == yr].copy()

    # keep valid rows
    df = df.dropna(subset=["T_REC_dt", "NOAA_AR"]).copy()

    # remove duplicate snapshot+AR pairs
    df = df.drop_duplicates(subset=["T_REC_dt", "NOAA_AR"], keep="first")

    # strip duplicate useless columns if they exist
    for col in ["USFLUX.1", "AREA_ACR.1"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    print(f"Final {yr} shape: {df.shape}")
    all_years.append(df)

full = pd.concat(all_years, ignore_index=True)
full = full.sort_values(["NOAA_AR", "T_REC_dt"]).reset_index(drop=True)

print("\n=== AFTER MERGE ===")
print("Shape:", full.shape)
print("\nCounts per year:")
print(full["year"].value_counts().sort_index())

print("\nAny duplicates (T_REC_dt + NOAA_AR)?")
print(full.duplicated(subset=["T_REC_dt", "NOAA_AR"]).sum())

full.to_csv(OUT_AR_CLEAN, index=False)
print("\nSaved canonical HMI base to:")
print(OUT_AR_CLEAN)

=== CLEAN 24H HMI BUILD STARTED ===

Loading 2010: hmi_2010_2010_fixed.csv
Final 2010 shape: (623, 29)

Loading 2011: hmi_2011_2011_fixed.csv
Final 2011 shape: (1865, 29)

Loading 2012: hmi_2012_2012_fixed.csv
Final 2012 shape: (1782, 29)

Loading 2013: hmi_2013_2013_fixed.csv
Final 2013 shape: (2267, 29)

Loading 2014: hmi_2014_2014_fixed.csv
Final 2014 shape: (2003, 29)

Loading 2015: hmi_2015_2015_fixed.csv
Final 2015 shape: (1837, 29)

Loading 2016: hmi_2016_2016_fixed.csv
Final 2016 shape: (1133, 29)

Loading 2017: hmi_2017_2017_fixed.csv
Final 2017 shape: (583, 29)

Loading 2018: hmi_2018_2018_fixed.csv
Final 2018 shape: (237, 29)

Loading 2019: hmi_2019_2019_fixed.csv
Final 2019 shape: (142, 29)

Loading 2020: hmi_2020_2020_fixed.csv
Final 2020 shape: (332, 29)

Loading 2021: hmi_2021_2021_fixed.csv
Final 2021 shape: (930, 29)

Loading 2022: hmi_2022_2022_fixed.csv
Final 2022 shape: (1977, 29)

Loading 2023: hmi_2023_2023_fixed.csv
Final 2023 shape: (2573, 29)

Loading 2024: hmi

# Cell 4 — create clean ML-ready 16-feature AR dataset

In [ ]:
# =========================================================
# 3. BUILD CLEAN ML-READY-16 DATASET
# =========================================================
available_core = [c for c in CORE16 if c in full.columns]
missing_core = [c for c in CORE16 if c not in full.columns]

print("Available CORE16:", available_core)
print("Missing CORE16:", missing_core)

if missing_core:
    raise ValueError(f"Missing required CORE16 columns: {missing_core}")

ar_base = full[available_core].copy()
ar_base = ar_base.dropna(subset=["T_REC_dt", "NOAA_AR"]).copy()
ar_base = ar_base.sort_values(["NOAA_AR", "T_REC_dt"]).reset_index(drop=True)

print("\nML-READY-16 shape:", ar_base.shape)
print("\nYear counts:")
print(ar_base["year"].value_counts().sort_index())

ar_base.to_csv(OUT_AR_ML16, index=False)
print("\nSaved clean ML-READY-16 base to:")
print(OUT_AR_ML16)

Available CORE16: ['T_REC_dt', 'year', 'NOAA_AR', 'MEANGBZ', 'MEANGAM', 'MEANGBT', 'MEANGBH', 'MEANJZD', 'TOTUSJZ', 'MEANALP', 'MEANJZH', 'ABSNJZH', 'SAVNCPP', 'MEANSHR', 'SHRGT45', 'R_VALUE', 'USFLUX', 'TOTPOT', 'TOTUSJH']
Missing CORE16: []

ML-READY-16 shape: (22355, 19)

Year counts:
year
2010     623
2011    1865
2012    1782
2013    2267
2014    2003
2015    1837
2016    1133
2017     583
2018     237
2019     142
2020     332
2021     930
2022    1977
2023    2573
2024    2619
2025    1452
Name: count, dtype: int64

Saved clean ML-READY-16 base to:
/content/drive/MyDrive/AR_Stratified/HMI_AR_2010_2025_ML_READY_16_CLEAN_24H_BASE.csv


# Cell 5 — GOES parser

In [ ]:
# =========================================================
# 4. GOES PARSER
# =========================================================
def parse_goes_line(line: str):
    line = line.strip()

    if not line:
        return None
    if "Written" in line or "GOES" in line or "events" in line.lower():
        return None

    tokens = line.split()
    if len(tokens) < 4:
        return None

    if not re.match(r"^\d{1,2}-[A-Za-z]{3}-\d{4}$", tokens[0]):
        return None

    date = tokens[0]

    class_idx = None
    for i, t in enumerate(tokens):
        if re.match(r"^[A-Z][0-9](\.[0-9])?$", t):
            class_idx = i
            break

    if class_idx is None:
        return None

    start_time = tokens[1] if len(tokens) > 1 else None
    peak_time = tokens[2] if len(tokens) > 2 else None
    flare_class = tokens[class_idx]
    location = tokens[class_idx + 1] if class_idx + 1 < len(tokens) else None

    noaa_ar = np.nan
    for t in tokens[class_idx + 1:]:
        if re.fullmatch(r"\d{4,5}", t):
            noaa_ar = int(t)
            break

    return {
        "flare_date": date,
        "start_time": start_time,
        "peak_time": peak_time,
        "flare_class": flare_class,
        "location": location,
        "NOAA_AR": noaa_ar,
    }


def get_flux_value(flare_class_str):
    if pd.isna(flare_class_str):
        return np.nan

    s = str(flare_class_str).strip()
    if len(s) < 2:
        return np.nan

    letter = s[0].upper()
    try:
        mag = float(s[1:])
    except ValueError:
        return np.nan

    if letter == "A":
        base_flux = 1e-8
    elif letter == "B":
        base_flux = 1e-7
    elif letter == "C":
        base_flux = 1e-6
    elif letter == "M":
        base_flux = 1e-5
    elif letter == "X":
        base_flux = 1e-4
    else:
        return np.nan

    return mag * base_flux

# Cell 6 — build clean GOES tables from raw text files

In [ ]:
# =========================================================
# 5. BUILD CLEAN GOES TABLES
# =========================================================
goes_files = sorted(glob.glob(f"{GOES_BASE}/GOES XRAY events for *.txt"))
print("GOES files found:", len(goes_files))

rows = []
for file in goes_files:
    print("Processing:", os.path.basename(file))
    with open(file, "r", errors="ignore") as f:
        for line in f:
            parsed = parse_goes_line(line)
            if parsed is not None:
                rows.append(parsed)

goes_df = pd.DataFrame(rows)
print("\nParsed GOES raw shape:", goes_df.shape)

goes_df = goes_df.dropna(subset=["flare_date", "start_time"]).copy()
goes_df["flare_start_dt"] = pd.to_datetime(
    goes_df["flare_date"] + " " + goes_df["start_time"],
    errors="coerce"
)
goes_df = goes_df.dropna(subset=["flare_start_dt"]).copy()

goes_df["flare_class"] = goes_df["flare_class"].astype(str).str.strip()
goes_df["class_letter"] = goes_df["flare_class"].str[0]
goes_df["class_mag"] = pd.to_numeric(goes_df["flare_class"].str[1:], errors="coerce")
goes_df["NOAA_AR"] = pd.to_numeric(goes_df["NOAA_AR"], errors="coerce").astype("Int64")
goes_df = goes_df.dropna(subset=["NOAA_AR", "class_letter"]).copy()
goes_df["year"] = goes_df["flare_start_dt"].dt.year
goes_df["peak_flux"] = goes_df["flare_class"].apply(get_flux_value)

# deduplicate to reduce artificial inflation
goes_df = goes_df.drop_duplicates(
    subset=["NOAA_AR", "flare_start_dt", "flare_class"],
    keep="first"
).reset_index(drop=True)

goes_all = goes_df.sort_values(["NOAA_AR", "flare_start_dt"]).reset_index(drop=True)
goes_mx = goes_all[goes_all["class_letter"].isin(["M", "X"])].copy()

print("\nGOES all shape:", goes_all.shape)
print("GOES M/X shape:", goes_mx.shape)

print("\nGOES all year counts:")
print(goes_all["year"].value_counts().sort_index())

print("\nGOES M/X year counts:")
print(goes_mx["year"].value_counts().sort_index())

GOES files found: 16
Processing: GOES XRAY events for 2010.txt
Processing: GOES XRAY events for 2011.txt
Processing: GOES XRAY events for 2012.txt
Processing: GOES XRAY events for 2013.txt
Processing: GOES XRAY events for 2014.txt
Processing: GOES XRAY events for 2015.txt
Processing: GOES XRAY events for 2016.txt
Processing: GOES XRAY events for 2017.txt
Processing: GOES XRAY events for 2018.txt
Processing: GOES XRAY events for 2019.txt
Processing: GOES XRAY events for 2020.txt
Processing: GOES XRAY events for 2021.txt
Processing: GOES XRAY events for 2022.txt
Processing: GOES XRAY events for 2023.txt
Processing: GOES XRAY events for 2024.txt
Processing: GOES XRAY events for 2025.txt

Parsed GOES raw shape: (34723, 6)

GOES all shape: (27465, 11)
GOES M/X shape: (2201, 11)

GOES all year counts:
year
2010     646
2011    1696
2012    1628
2013    1235
2014    1808
2015    1599
2016    1112
2017     888
2018     323
2019     277
2020     681
2021    1946
2022    3487
2023    4421
2024  

### **Detailed Explanation: Building the 5 Flare-History Features (Code Walkthrough for Cell 7)**

This section explains, with direct code examples, how the 5 crucial flare-history features are constructed for each observation of an active region. This process involves looking backward in time from each `T_REC_dt` (observation timestamp) to gather information about past flare activity.

First, we initialize the new columns in our main `ar_24h` DataFrame, setting default values (like 0 for counts or `NaN` for numerical features that haven't occurred yet).

In [ ]:
ar_24h = ar_base.copy()

# Initialize output columns
ar_24h["label_MX_1d"] = 0
ar_24h["future_flare_start_dt_mx1d"] = pd.NaT
ar_24h["future_flare_peak_time_mx1d"] = pd.Series(index=ar_24h.index, dtype="object")
ar_24h["future_max_class_mx1d"] = pd.Series(index=ar_24h.index, dtype="object")

ar_24h["flare_count_past_24h"] = 0
ar_24h["time_since_last_flare_hours_24h"] = np.nan
ar_24h["max_peak_flux_past_24h"] = np.nan
ar_24h["mean_peak_flux_past_24h"] = np.nan
ar_24h["flare_activity_index_past_24h"] = 0.0
ar_24h["had_flare_past_24h"] = 0

The core of the feature construction happens in a loop that iterates through each unique `NOAA_AR` (Active Region number). For each active region, we filter the `goes_all` DataFrame to get only the flares associated with that specific AR. Then, for every observation (`T_REC_dt`) within that AR's history, we define a 24-hour look-back window and calculate the features.

Let's focus on the `Past 24h features` loop:

```python
for ar_num in tqdm(unique_ars, desc="Past 24h features"):
    ar_idx = ar_24h[ar_24h["NOAA_AR"] == ar_num].index
    gsub = goes_all[goes_all["NOAA_AR"] == ar_num].copy()

    if gsub.empty:
        continue

    for idx in ar_idx:
        current_time = ar_24h.loc[idx, "T_REC_dt"]
        window_start = current_time - pd.Timedelta(hours=24)

        # ... calculations for each feature ...
```

Here's what each part does:

*   `current_time = ar_24h.loc[idx, "T_REC_dt"]`: This gets the exact timestamp of the current active region observation we are processing.
*   `window_start = current_time - pd.Timedelta(hours=24)`: This defines the beginning of our 24-hour look-back window. So, for any `current_time`, we are only interested in flares that started after `window_start` and before `current_time`.

### **Identifying Relevant Flares**

For each `current_time`, we filter the flares associated with the active region (`gsub`) to find only those that occurred within our 24-hour look-back window:

In [ ]:
relevant_flares_past = gsub[
    (gsub["flare_start_dt"] > window_start) &
    (gsub["flare_start_dt"] < current_time)
]

This `relevant_flares_past` DataFrame contains *all flares that started in the 24 hours immediately preceding the `current_time`* for the specific `NOAA_AR`.

### **Calculating the 5 Flare-History Features**

Now, we use `relevant_flares_past` to compute the features:

1.  **`flare_count_past_24h` (Number of Flares in past 24h)**
    *   **Code:** `ar_24h.loc[idx, "flare_count_past_24h"] = len(relevant_flares_past)`
    *   **Explanation:** We simply count the number of rows (flares) in the `relevant_flares_past` DataFrame. If it's empty, the count is 0.

2.  **`max_peak_flux_past_24h` (Maximum Peak Flux in past 24h)**
    *   **Code:** `ar_24h.loc[idx, "max_peak_flux_past_24h"] = (relevant_flares_past["peak_flux"].max() if not relevant_flares_past.empty else np.nan)`
    *   **Explanation:** We take the `peak_flux` column from `relevant_flares_past` and find its maximum value. If there were no flares in the window, it's set to `NaN`.

3.  **`mean_peak_flux_past_24h` (Mean Peak Flux in past 24h)**
    *   **Code:** `ar_24h.loc[idx, "mean_peak_flux_past_24h"] = (relevant_flares_past["peak_flux"].mean() if not relevant_flares_past.empty else np.nan)`
    *   **Explanation:** Similar to the max, but we calculate the average of the `peak_flux` values for all flares in `relevant_flares_past`. If no flares, it's `NaN`.

4.  **`flare_activity_index_past_24h` (Sum of Peak Fluxes in past 24h)**
    *   **Code:** `ar_24h.loc[idx, "flare_activity_index_past_24h"] = (relevant_flares_past["peak_flux"].sum() if not relevant_flares_past.empty else 0.0)`
    *   **Explanation:** We sum all `peak_flux` values in `relevant_flares_past`. If no flares, the sum is `0.0` (indicating no activity).

5.  **`time_since_last_flare_hours_24h` (Hours since the last flare before `current_time`)**
    *   **Code:**
    ```python
    previous_flares = gsub[gsub["flare_start_dt"] < current_time]
    if not previous_flares.empty:
        last_flare_time = previous_flares["flare_start_dt"].max()
        dt_hours = (current_time - last_flare_time).total_seconds() / 3600.0
        ar_24h.loc[idx, "time_since_last_flare_hours_24h"] = dt_hours
    ```
    *   **Explanation:** This one is slightly different. Instead of the 24-hour window, we look at *all* flares for the `NOAA_AR` that occurred *before* the `current_time`. We find the latest one (`.max()`), calculate the time difference in hours, and store it. If no flares occurred before `current_time`, it remains `NaN`.

Finally, we have a binary feature to quickly check if any flare occurred in the past 24 hours:

*   **`had_flare_past_24h` (Binary: 1 if flare in past 24h, 0 otherwise)**
    *   **Code:** `ar_24h["had_flare_past_24h"] = (ar_24h["flare_count_past_24h"] > 0).astype(int)`
    *   **Explanation:** This is derived directly from `flare_count_past_24h`. If the count is greater than 0, it means a flare happened, so it's `1`; otherwise, it's `0`.

This detailed breakdown of the code directly shows how each of these past flare-history features is calculated for every single observation in the dataset, always respecting the time causality (only looking into the past).

# Cell 7 — construct 24h labels and past-24h flare-history features from scratch

In [ ]:
# =========================================================
# 6. BUILD 24H LABELS + PAST-24H FEATURES FROM SCRATCH
# =========================================================
ar_24h = ar_base.copy()

# initialize output columns
ar_24h["label_MX_1d"] = 0
ar_24h["future_flare_start_dt_mx1d"] = pd.NaT
ar_24h["future_flare_peak_time_mx1d"] = pd.Series(index=ar_24h.index, dtype="object")
ar_24h["future_max_class_mx1d"] = pd.Series(index=ar_24h.index, dtype="object")

ar_24h["flare_count_past_24h"] = 0
ar_24h["time_since_last_flare_hours_24h"] = np.nan
ar_24h["max_peak_flux_past_24h"] = np.nan
ar_24h["mean_peak_flux_past_24h"] = np.nan
ar_24h["flare_activity_index_past_24h"] = 0.0
ar_24h["had_flare_past_24h"] = 0

ar_24h = ar_24h.sort_values(["NOAA_AR", "T_REC_dt"]).reset_index(drop=True)
goes_all = goes_all.sort_values(["NOAA_AR", "flare_start_dt"]).reset_index(drop=True)
goes_mx = goes_mx.sort_values(["NOAA_AR", "flare_start_dt"]).reset_index(drop=True)

# ----- 6A. build past-24h flare-history features -----
print("Calculating past-24h flare-history features...")
unique_ars = ar_24h["NOAA_AR"].dropna().unique()

for ar_num in tqdm(unique_ars, desc="Past 24h features"):
    ar_idx = ar_24h[ar_24h["NOAA_AR"] == ar_num].index
    gsub = goes_all[goes_all["NOAA_AR"] == ar_num].copy()

    if gsub.empty:
        continue

    for idx in ar_idx:
        current_time = ar_24h.loc[idx, "T_REC_dt"]
        window_start = current_time - pd.Timedelta(hours=24)

        relevant_flares_past = gsub[
            (gsub["flare_start_dt"] > window_start) &
            (gsub["flare_start_dt"] < current_time)
        ]

        previous_flares = gsub[gsub["flare_start_dt"] < current_time]

        ar_24h.loc[idx, "flare_count_past_24h"] = len(relevant_flares_past)
        ar_24h.loc[idx, "max_peak_flux_past_24h"] = (
            relevant_flares_past["peak_flux"].max() if not relevant_flares_past.empty else np.nan
        )
        ar_24h.loc[idx, "mean_peak_flux_past_24h"] = (
            relevant_flares_past["peak_flux"].mean() if not relevant_flares_past.empty else np.nan
        )
        ar_24h.loc[idx, "flare_activity_index_past_24h"] = (
            relevant_flares_past["peak_flux"].sum() if not relevant_flares_past.empty else 0.0
        )

        if not previous_flares.empty:
            last_flare_time = previous_flares["flare_start_dt"].max()
            dt_hours = (current_time - last_flare_time).total_seconds() / 3600.0
            ar_24h.loc[idx, "time_since_last_flare_hours_24h"] = dt_hours

ar_24h["had_flare_past_24h"] = (ar_24h["flare_count_past_24h"] > 0).astype(int)

# ----- 6B. build 24h future labels -----
print("Applying future 24h MX labels...")

def choose_best(old, new):
    if pd.isna(old):
        return new

    order = {"X": 3, "M": 2, "C": 1, "B": 0, "A": 0}

    if old[0] == new[0]:
        try:
            return old if float(old[1:]) >= float(new[1:]) else new
        except Exception:
            return old

    return old if order.get(old[0], 0) >= order.get(new[0], 0) else new

for _, row in tqdm(goes_mx.iterrows(), total=len(goes_mx), desc="Future MX labels"):
    ar_num = row["NOAA_AR"]
    flare_time = row["flare_start_dt"]
    flare_cls = row["flare_class"]
    peak_time = row.get("peak_time", None)

    start_win = flare_time - pd.Timedelta(hours=24)
    end_win = flare_time

    mask = (
        (ar_24h["NOAA_AR"] == ar_num) &
        (ar_24h["T_REC_dt"] >= start_win) &
        (ar_24h["T_REC_dt"] < end_win)
    )

    if not mask.any():
        continue

    ar_24h.loc[mask, "label_MX_1d"] = 1
    ar_24h.loc[mask, "future_flare_start_dt_mx1d"] = flare_time
    ar_24h.loc[mask, "future_flare_peak_time_mx1d"] = peak_time

    current_vals = ar_24h.loc[mask, "future_max_class_mx1d"]
    ar_24h.loc[mask, "future_max_class_mx1d"] = [
        choose_best(old, flare_cls) for old in current_vals
    ]

print("\nFinal 24h shape:", ar_24h.shape)
print("\nLabel distribution:")
print(ar_24h["label_MX_1d"].value_counts())

print("\nValue counts for had_flare_past_24h:")
print(ar_24h["had_flare_past_24h"].value_counts())

Calculating past-24h flare-history features...


Past 24h features:   0%|          | 0/2268 [00:00<?, ?it/s]

Applying future 24h MX labels...


Future MX labels:   0%|          | 0/2201 [00:00<?, ?it/s]


Final 24h shape: (22355, 29)

Label distribution:
label_MX_1d
0    21594
1      761
Name: count, dtype: int64

Value counts for had_flare_past_24h:
had_flare_past_24h
0    16818
1     5537
Name: count, dtype: int64


# Cell 8 — leakage checks

In [ ]:
# =========================================================
# 7. LEAKAGE CHECKS
# =========================================================
print("\n--- Leakage Verification: label_MX_1d ---")
positive_labels = ar_24h[ar_24h["label_MX_1d"] == 1].copy()

leakage_rows_label = positive_labels[
    positive_labels["future_flare_start_dt_mx1d"].notna() &
    (positive_labels["future_flare_start_dt_mx1d"] <= positive_labels["T_REC_dt"])
]

if not leakage_rows_label.empty:
    print(f"⚠ Found {len(leakage_rows_label)} possible label leakage rows.")
    display(leakage_rows_label[[
        "T_REC_dt", "NOAA_AR", "label_MX_1d", "future_flare_start_dt_mx1d"
    ]].head())
else:
    print("✅ No label leakage detected for label_MX_1d.")

print("\n--- Leakage Verification: past-24h features ---")
hist_leak_found = False

rows_with_past = ar_24h[ar_24h["flare_count_past_24h"] > 0].copy()

for _, r in tqdm(rows_with_past.iterrows(), total=len(rows_with_past), desc="Checking past-feature leakage"):
    current_time = r["T_REC_dt"]
    ar_num = r["NOAA_AR"]
    window_start = current_time - pd.Timedelta(hours=24)

    gsub = goes_all[goes_all["NOAA_AR"] == ar_num].copy()
    actual_relevant = gsub[
        (gsub["flare_start_dt"] > window_start) &
        (gsub["flare_start_dt"] < current_time)
    ]

    bad = actual_relevant[actual_relevant["flare_start_dt"] >= current_time]
    if not bad.empty:
        print(f"⚠ Historical leakage found for AR {ar_num} at {current_time}")
        display(bad.head())
        hist_leak_found = True
        break

if not hist_leak_found:
    print("✅ No historical leakage detected for past-24h features.")


--- Leakage Verification: label_MX_1d ---
✅ No label leakage detected for label_MX_1d.

--- Leakage Verification: past-24h features ---


Checking past-feature leakage:   0%|          | 0/5537 [00:00<?, ?it/s]

✅ No historical leakage detected for past-24h features.


# Cell 9 — provenance and sanity summary

In [ ]:
# =========================================================
# 8. PROVENANCE + SANITY SUMMARY
# =========================================================
print("Raw canonical HMI base shape:", full.shape)
print("ML-ready 16 shape:", ar_base.shape)
print("Final 24h dataset shape:", ar_24h.shape)

print("\nYear counts in final 24h dataset:")
print(ar_24h["year"].value_counts().sort_index())

print("\nDuplicates in final 24h dataset by (T_REC_dt, NOAA_AR):")
print(ar_24h.duplicated(subset=["T_REC_dt", "NOAA_AR"]).sum())

print("\nMissing values in key columns:")
check_cols = [
    "T_REC_dt", "NOAA_AR", "label_MX_1d",
    "flare_count_past_24h", "time_since_last_flare_hours_24h",
    "max_peak_flux_past_24h", "mean_peak_flux_past_24h",
    "flare_activity_index_past_24h"
]
print(ar_24h[check_cols].isna().sum())

Raw canonical HMI base shape: (22355, 29)
ML-ready 16 shape: (22355, 19)
Final 24h dataset shape: (22355, 29)

Year counts in final 24h dataset:
year
2010     623
2011    1865
2012    1782
2013    2267
2014    2003
2015    1837
2016    1133
2017     583
2018     237
2019     142
2020     332
2021     930
2022    1977
2023    2573
2024    2619
2025    1452
Name: count, dtype: int64

Duplicates in final 24h dataset by (T_REC_dt, NOAA_AR):
0

Missing values in key columns:
T_REC_dt                               0
NOAA_AR                                0
label_MX_1d                            0
flare_count_past_24h                   0
time_since_last_flare_hours_24h    11217
max_peak_flux_past_24h             16818
mean_peak_flux_past_24h            16818
flare_activity_index_past_24h          0
dtype: int64


# Cell 10 — save final canonical 24h dataset

In [ ]:
# =========================================================
# 9. SAVE FINAL CANONICAL 24H DATASET
# =========================================================
ar_24h.to_csv(OUT_24H_FINAL, index=False)
print("Saved final canonical 24h dataset to:")
print(OUT_24H_FINAL)

print("\nSample rows:")
display(ar_24h[[
    "T_REC_dt", "NOAA_AR", "label_MX_1d",
    "flare_count_past_24h", "had_flare_past_24h",
    "time_since_last_flare_hours_24h",
    "max_peak_flux_past_24h", "mean_peak_flux_past_24h",
    "flare_activity_index_past_24h",
    "future_flare_start_dt_mx1d"
]].head(10))

Saved final canonical 24h dataset to:
/content/drive/MyDrive/AR_Stratified/HMI_AR_2010_2025_ML_READY_16_LABELED_MX1D_FLARE_FEATURES_24H_CLEAN.csv

Sample rows:


,T_REC_dt,NOAA_AR,label_MX_1d,flare_count_past_24h,had_flare_past_24h,time_since_last_flare_hours_24h,max_peak_flux_past_24h,mean_peak_flux_past_24h,flare_activity_index_past_24h,future_flare_start_dt_mx1d
0,2010-05-03 12:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
1,2010-05-04 12:00:00,11063,0,1,1,22.366667,1.500000e-07,1.500000e-07,1.500000e-07,NaT
2,2010-05-05 12:00:00,11063,0,0,0,46.366667,NaN,NaN,0.000000e+00,NaT
3,2010-05-01 12:00:00,11064,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
4,2010-05-02 12:00:00,11064,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
5,2010-05-03 12:00:00,11064,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
6,2010-05-04 12:00:00,11064,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
7,2010-05-05 12:00:00,11064,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
8,2010-05-01 12:00:00,11065,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
9,2010-05-02 12:00:00,11065,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT


# STEP 1 — Clean the dataset (VERY IMPORTANT)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/AR_Stratified/HMI_AR_2010_2025_ML_READY_16_LABELED_MX1D_FLARE_FEATURES_24H_CLEAN.csv")

# DROP ALL FUTURE COLUMNS (MANDATORY)
df = df.drop(columns=[
    'future_flare_start_dt_mx1d',
    'future_flare_peak_time_mx1d',
    'future_max_class_mx1d'
], errors='ignore')

print("Shape after cleaning:", df.shape)

Shape after cleaning: (22355, 26)


# STEP 2 — Sanity checks

In [ ]:
print("\nDuplicates:")
print(df.duplicated(subset=['T_REC_dt','NOAA_AR']).sum())

print("\nLabel distribution:")
print(df['label_MX_1d'].value_counts())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))


Duplicates:
0

Label distribution:
label_MX_1d
0    21594
1      761
Name: count, dtype: int64

Missing values:
mean_peak_flux_past_24h            16818
max_peak_flux_past_24h             16818
time_since_last_flare_hours_24h    11217
MEANSHR                              217
MEANGAM                              177
MEANGBH                              173
MEANALP                              173
MEANJZD                              173
MEANGBT                              173
MEANGBZ                              173
dtype: int64


# STEP 3 — Confirm temporal logic

In [ ]:
df = df.sort_values(['NOAA_AR','T_REC_dt']).reset_index(drop=True)

print(df[['T_REC_dt','label_MX_1d']].head(10))

              T_REC_dt  label_MX_1d
0  2010-05-03 12:00:00            0
1  2010-05-04 12:00:00            0
2  2010-05-05 12:00:00            0
3  2010-05-01 12:00:00            0
4  2010-05-02 12:00:00            0
5  2010-05-03 12:00:00            0
6  2010-05-04 12:00:00            0
7  2010-05-05 12:00:00            0
8  2010-05-01 12:00:00            0
9  2010-05-02 12:00:00            0


# STEP 4 — Freeze the dataset

In [ ]:
df.to_csv("24H_FINAL_CLEAN_READY.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/AR_Stratified/24H_FINAL_CLEAN_READY.csv")

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

print("\nLabel distribution:")
print(df['label_MX_1d'].value_counts())

print("\nCheck duplicates:")
print(df.duplicated(subset=['T_REC_dt','NOAA_AR']).sum())

print("\nMissing values (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

Shape: (22355, 26)

Columns:
 ['T_REC_dt', 'year', 'NOAA_AR', 'MEANGBZ', 'MEANGAM', 'MEANGBT', 'MEANGBH', 'MEANJZD', 'TOTUSJZ', 'MEANALP', 'MEANJZH', 'ABSNJZH', 'SAVNCPP', 'MEANSHR', 'SHRGT45', 'R_VALUE', 'USFLUX', 'TOTPOT', 'TOTUSJH', 'label_MX_1d', 'flare_count_past_24h', 'time_since_last_flare_hours_24h', 'max_peak_flux_past_24h', 'mean_peak_flux_past_24h', 'flare_activity_index_past_24h', 'had_flare_past_24h']

Label distribution:
label_MX_1d
0    21594
1      761
Name: count, dtype: int64

Check duplicates:
0

Missing values (top 10):
mean_peak_flux_past_24h            16818
max_peak_flux_past_24h             16818
time_since_last_flare_hours_24h    11217
MEANSHR                              217
MEANGAM                              177
MEANGBH                              173
MEANALP                              173
MEANJZD                              173
MEANGBT                              173
MEANGBZ                              173
dtype: int64


In [ ]:
# Make sure NO future columns exist
for col in df.columns:
    if "future" in col.lower():
        print("⚠ FOUND FUTURE COLUMN:", col)